In [ ]:
JOBS_GPU0 = ['icarl_full Gram seed0']
JOBS_GPU1 = ['Gram verification; CG-IR profile']
JOBS_CPU = []
RUN_TAG = 'vna_v4_gram_m1s0_ir'
PRESET = 'GRAM_M1S0_IR_PROFILE'
REPO_COMMIT = '3a6638846b612b29dcc8c4010ced759a46b151a2'
SOURCE_DIGEST = None
GRAD_CLIP = None  # Registered runs stay unclipped; selected tau is authorized only by the clip-study C gate.
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000


# Gram M1s0 + CG-IR profile — current policy, T4 × 2
Attach CIFAR-100, the d88abc4 full task-1 checkpoint, and old baseline/output archives.
Both active lanes use the same immutable source commit and dead-only Schur/rho>=1/4 policy.
The 8c56d02 checkout checks fixed-system Gram direction math only; changed
LM/GIR decisions are not compared to that old policy.

GPU0 calibrates Gram/power10 K0 from three task-2 points while GPU1 profiles
current Gram on the shared checkpoint and checks the fixed system against 8c56d02.
Full Gram M1s0 starts only after the current profile completes and that check passes. GPU1 then
profiles CG-IR/exact with capacity measurements and stops; it never trains M1.
After full completes, GPU0 checks the old baseline against a fresh 150-epoch
baseline prefix; exact loss equality permits historical result reuse, otherwise
training continues to task 20. Missing old baseline runs a fresh full baseline.
Outputs/checkpoints resume from attached archives; archives update every 15 min.
Dead/dormant probe=256; B uses CG-IR on the first 128 of those samples.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_gram_m1s0_ir")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(900):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from experiments.io_utils import atomic_json_save
from diagnostics.exact_solver import find_task1_checkpoint, compare_profiles
from experiments.main_cil_baseline import prepare_reference
from experiments.runners.pycil import upstream_imports
ROOT=STAGE_B/RUN_TAG/REPO_COMMIT[:12]
ROOT.mkdir(parents=True,exist_ok=True)
if torch.cuda.device_count()!=2 or any('T4' not in torch.cuda.get_device_name(i) for i in range(2)):
    raise RuntimeError('This notebook requires T4 x2')
ENV['VNA_DIAGNOSTIC_LANES']='2'
REFERENCE_COMMIT='8c56d02dab1ee16c47eb9138bf4aa3a64b9b45e2'
REFERENCE_REPO=REPO.with_name('counterfactual-projection-gram-8c56d02')
if REFERENCE_REPO.exists():
    head=subprocess.check_output(['git','-C',str(REFERENCE_REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REFERENCE_REPO),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if head!=REFERENCE_COMMIT or dirty:shutil.rmtree(REFERENCE_REPO)
if not REFERENCE_REPO.exists():private_clone(MAIN_URL,REFERENCE_REPO,REFERENCE_COMMIT)
for upstream in (REPO/'third_party').iterdir():
    destination=REFERENCE_REPO/'third_party'/upstream.name
    if upstream.is_dir() and not destination.exists():destination.symlink_to(upstream,target_is_directory=True)
upstream_imports()
from experiments.main_cil_metric_resume import restore_metric_trace_run
restore_metric_trace_run(STAGE_B,RUN_TAG,REPO_COMMIT)
checkpoint=find_task1_checkpoint(Path('/kaggle/input'),ROOT/'profiles'/'task1_source.pt')
script=REPO/'diagnostics/exact_solver.py'
gate=threading.Event()
gram_check={}
main_common=[sys.executable,'-u','-m','experiments.main_cil','--solver','gram','--curvature','power10',
    '--seed','0','--backbone','resnet32_nobn','--class-order','superclass','--nobn-lr-scale','.5',
    '--device','cuda:0','--max-tasks','20','--data-dir',str(DATA_ROOT)]

def profile(name,tree,solver,curvature,capacity=False,reference=False):
    out=ROOT/'profiles'/name;out.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,'-u',str(script),'--repo',str(tree),'--branch','D','--profile',
        '--solver',solver,'--curvature',curvature,'--checkpoint',str(checkpoint),
        '--data-dir',str(DATA_ROOT),'--out',str(out)]
    if capacity:cmd+=['--capacity-profile']
    if reference:cmd+=['--gram-reference',str(REFERENCE_REPO)]
    if not launch(cmd,1,name,RUN_ROOT/(name+'.log')):return None
    archive_outputs()
    return json.loads((out/'diagnostic.json').read_text())

def run_gpu1():
    try:
        current=profile('gram_current',REPO,'gram','power10',reference=True)
        if current is None:return
        result=dict(current.get('fixed_gram_reference',dict(status='INCOMPLETE')))
        if current.get('status')!='PROFILE_COMPLETED':result.update(status='FAIL',reason='Current Gram profile incomplete')
        result['profile_identity']=current.get('identity')
        atomic_json_save(result,ROOT/'profiles'/'gram_compatibility.json')
        gram_check.update(result)
        if result['status']!='PASS':raise RuntimeError('Fixed-system Gram check must PASS before full training')
    finally:
        gate.set()
        archive_outputs()
    try:
        cheap=profile('cg_ir',REPO,'cg_ir','exact',capacity=True)
        if cheap is None:return
    except Exception as error:
        path=ROOT/'profiles'/'cg_ir'/'diagnostic.json'
        cheap=json.loads(path.read_text()) if path.exists() else dict(status='FAILED_DIAGNOSTIC',error=str(error))
        print('CG-IR profile failed; the approved Gram training lane continues:',error,flush=True)
    result=compare_profiles(current,cheap,strict=True,allow_curvature_change=True)
    if result['status']=='INCOMPLETE':
        result.update(status='FAIL',comparison_status='INCOMPLETE',reason='Profile did not complete')
    result.setdefault('lm_numeric_audit',dict(status='UNAVAILABLE',reason='Profile incomplete'))
    atomic_json_save(result,ROOT/'profiles'/'cg_ir_vs_gram.json')
    print('CG-IR versus same-commit Gram:',json.dumps(result,indent=2),flush=True)
    lane_state[1]={'job':'idle','last':'CG-IR profile finished; no cheap M1 authorized'}
    archive_outputs()

def run_gpu0():
    schedule=ROOT/'_schedules'/'gram_power10.json'
    if not schedule.exists():
        cmd=main_common+['--arm','icarl_full','--preflight-only','--schedule-output',str(schedule),
            '--out',str(ROOT/'_preflight'/'gram')]
        if not launch(cmd,0,'gram_preflight',RUN_ROOT/'gram_preflight.log'):return
        archive_outputs()
    while not gate.wait(1):
        if pause_due():return
    if pause_due():return
    if gram_check.get('status')!='PASS':raise RuntimeError('Gram reference check failed; full training held')
    cmd=main_common+['--arm','icarl_full','--schedule',str(schedule),'--out',str(ROOT/'icarl_full')]
    if not launch(cmd,0,'gram_full_s0',RUN_ROOT/'gram_full_s0.log'):return
    archive_outputs()
    full=json.loads((ROOT/'icarl_full'/'summary.json').read_text())
    if full.get('status')!='COMPLETED':raise RuntimeError('Gram full did not finish; baseline remains queued')
    baseline=ROOT/'icarl';reference=ROOT/'baseline_reference.json'
    if reference.exists() or not (baseline/'latest.pt').exists():
        try:prepare_reference((STAGE_B,Path('/kaggle/input')),reference)
        except ValueError as error:print('Old baseline cannot be reused; run fresh:',error,flush=True)
    cmd=main_common+['--arm','icarl','--out',str(baseline)]
    if reference.exists():cmd+=['--baseline-reference',str(reference)]
    if not pause_due():launch(cmd,0,'gram_baseline_s0',RUN_ROOT/'gram_baseline_s0.log')
    archive_outputs()

failures=[]
try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(run_gpu0),pool.submit(run_gpu1)]
        for future in as_completed(futures):
            try:future.result()
            except Exception as error:failures.append(error)
finally:
    archive_stop.set();archiver.join(timeout=10);archive_outputs()
if failures:raise failures[0]
if archive_errors:raise RuntimeError(archive_errors)


In [ ]:
for name in ('gram_compatibility.json','cg_ir_vs_gram.json'):
    path=ROOT/'profiles'/name
    if path.exists():print(name,path.read_text())
for name in ('icarl_full','icarl'):
    path=ROOT/name/'summary.json'
    if path.exists():
        value=json.loads(path.read_text())
        if value.get('status')=='REUSED_BASELINE':
            reused=json.loads((path.parent/'reused_baseline.json').read_text())
            value=reused['reference']['summary']
            print('Historical baseline reused after exact 150-epoch loss verification:',reused['verification'])
        print(name,{k:value.get(k) for k in ('status','tasks_run','seconds','cnn_average_accuracy','nme_average_accuracy')})
path=ROOT/'profiles'/'cg_ir'/'diagnostic.json'
if path.exists():
    value=json.loads(path.read_text());event=(value.get('applications') or [{}])[0]
    print(json.dumps({**{k:value.get(k) for k in ('status','application_seconds','application_peak_cuda_bytes','post_application_sgd',
        'capacity_profile','gamma_precision','ir_summary','timers','final_L_KD')},
        'downstream':event.get('downstream'),'solves':value.get('cg')},indent=2))
archive_outputs()
print('GPU1 has no further jobs. CG-IR M1 remains unapproved and was not launched.')
